# Libraries

In [1]:
import os
import numpy as np
import torch
import onnx
import onnxruntime
from cardionix.models import CardioNetV2

# Functions

In [2]:
def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    return tensor.detach().cpu().numpy() \
        if tensor.requires_grad \
        else tensor.cpu().numpy()

# Paths

In [3]:
DIR_PATH: str = "../weights"
_WEIGHTS_cardionetv2_multi: str = os.path.join(DIR_PATH, "multi", "cardionetv2multi.pt")
_WEIGHTS_cardionetv2uno: str = os.path.join(DIR_PATH, "uno", "cardionetv2uno.pt")

# Load models

## Multi

In [4]:
multi = CardioNetV2(
    num_classes=3,
    audio_features_shape=(431, 128),
    tabular_features=50,
    rnn_layers=1,
    resnet_backbone={
        512: 2,
        1024: 2,
        2048: 2,
    },
    mixer_depth={
        "tabular": [128, 256, 512],
        "mixer": [2048]
    }
)

state_dict = torch.load(_WEIGHTS_cardionetv2_multi)
multi.load_state_dict(state_dict)
multi.eval()

CardioNetV2(
  (rnn): LSTM(128, 213, batch_first=True, bidirectional=True)
  (resnet): ResNet(
    (stem): Sequential(
      (0): Conv1d(431, 512, kernel_size=(7,), stride=(2,))
      (1): MaxPool1d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (residual_groups): Sequential(
      (0): Sequential(
        (0): ResidualUnite(
          (shortcut): Sequential()
          (activation): ReLU()
          (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv1): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,))
          (bn2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,))
        )
        (1): ResidualUnite(
          (shortcut): Sequential()
          (activation): ReLU()
          (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv1):

## Uno

In [4]:
uno = CardioNetV2(
    num_classes=3,
    audio_features_shape=(431, 128),
    rnn_hidden=256,
    rnn_layers=1,
    resnet_backbone={
        512: 2,
        1024: 2,
        2048: 2,
    },
    mixer_depth={
        "mixer": [1024]
    }
)

state_dict = torch.load(_WEIGHTS_cardionetv2uno)
uno.load_state_dict(state_dict)
uno.eval()

CardioNetV2(
  (rnn): LSTM(128, 256, batch_first=True, bidirectional=True)
  (resnet): ResNet(
    (stem): Sequential(
      (0): Conv1d(431, 512, kernel_size=(7,), stride=(2,))
      (1): MaxPool1d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (residual_groups): Sequential(
      (0): Sequential(
        (0): ResidualUnite(
          (shortcut): Sequential()
          (activation): ReLU()
          (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv1): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,))
          (bn2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,))
        )
        (1): ResidualUnite(
          (shortcut): Sequential()
          (activation): ReLU()
          (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv1):

# Export to onnx

## Multi-modal cardionetV2

In [10]:
# Input to the model
audio = torch.randn(1, 431, 128, requires_grad=True)
tabular = torch.randn(1, 50, requires_grad=True)
out = multi(audio, tabular)

# Export the model
torch.onnx.export(multi,                       # model being run
                  args=(audio, tabular),                         # model input (or a tuple for multiple inputs)
                  f="cardionetv2multi.onnx",     # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  opset_version=17,          # the ONNX version to export the model to
                  do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['audio', 'tabular'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}}
                  )

/home/merci/PycharmProjects/pet-projects/cardio-sonix-pipeline/env/lib64/python3.10/site-packages/torch/onnx/symbolic_opset9.py:4476: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================


## Test inference

In [18]:
onnx_model = onnx.load("cardionetv2multi.onnx")
onnx.checker.check_model(onnx_model)
session = onnxruntime.InferenceSession("cardionetv2multi.onnx", providers=["CPUExecutionProvider"])

In [20]:
session.get_inputs()[1].name

'tabular'

In [21]:
out

tensor([ 3.3727, -2.5457, -1.1044], grad_fn=<AddBackward0>)

In [22]:
# compute ONNX Runtime output prediction
inputs = {
    "audio": to_numpy(audio),
    "tabular": to_numpy(tabular)
}

outputs = session.run(None, inputs)
outputs

[array([ 3.3726852, -2.545749 , -1.1044222], dtype=float32)]

In [24]:
np.testing.assert_allclose(to_numpy(out), outputs[0], rtol=1e-03, atol=1e-05)

## Uni-modal cardionetV2

In [5]:
# Input to the model
x = torch.randn(1, 431, 128, requires_grad=True)
out = uno(x)

# Export the model
torch.onnx.export(uno,                       # model being run
                  x,                         # model input (or a tuple for multiple inputs)
                  "cardionetv2uno.onnx",     # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  opset_version=17,          # the ONNX version to export the model to
                  do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['input'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}}
                  )

/home/merci/PycharmProjects/pet-projects/cardio-sonix-pipeline/env/lib64/python3.10/site-packages/torch/onnx/symbolic_opset9.py:4476: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================


## Test inference

In [6]:
onnx_model = onnx.load("cardionetv2uno.onnx")
onnx.checker.check_model(onnx_model)
session = onnxruntime.InferenceSession("cardionetv2uno.onnx", providers=["CPUExecutionProvider"])

In [7]:
x = torch.randn(1, 431, 128)
y = uno(x)
y

tensor([[ 2.5230, -1.4508, -1.3373]], grad_fn=<AddmmBackward0>)

In [10]:
# compute ONNX Runtime output prediction
inputs = {
    session.get_inputs()[0].name: to_numpy(x)
}

outputs = session.run(None, inputs)
outputs

[array([[ 2.5230196, -1.4507648, -1.3373346]], dtype=float32)]

In [11]:
np.testing.assert_allclose(to_numpy(y), outputs[0], rtol=1e-03, atol=1e-05)